# 06 — Evaluate on the TEST split (run after training finishes)

Loads the best checkpoint of the **`faces_yolo26n_v1`** run and evaluates it on the
**independent test split** (leakage-free, never seen during training). Then it checks the
per-class **Recall / Precision** against the project's target metrics and prints a clear
**PASS / FAIL** per category.

**Targets** (in case of doubt, recall is prioritised over precision — a missed
face/plate is a privacy leak):
- `face`: Recall ≥ 0.95 AND Precision ≥ 0.85
- `license-plate`: Recall ≥ 0.90 (MVP) AND Precision ≥ 0.90 (production goal: Recall ≥ 0.98)

Outputs a persistent report to `runs/faces/faces_yolo26n_v1/eval_report.{json,md}` and
the usual confusion-matrix / PR curves under `runs/faces/faces_yolo26n_v1_TEST/`.

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/shared/s0598584/scripts')
import piheif_fix  # noqa: F401
import os
os.environ['YOLO_CONFIG_DIR'] = '/home/jovyan/shared/s0598584/ultralytics_cfg'
from pathlib import Path
from ultralytics import YOLO

ROOT = Path('/home/jovyan/shared/s0598584')
YAML_PATH = ROOT/'dataset_face_lp'/'dataset.yaml'
RUN_DIR = ROOT/'runs'/'faces'/'faces_yolo26n_v1'
WEIGHTS = RUN_DIR/'weights'
CKPT = WEIGHTS/'best.pt' if (WEIGHTS/'best.pt').exists() else WEIGHTS/'last.pt'
assert CKPT.exists(), f'no checkpoint in {WEIGHTS} — has training finished?'
assert YAML_PATH.exists(), f'dataset.yaml missing: {YAML_PATH}'
best = YOLO(str(CKPT))
print('loaded:', CKPT)
print('classes:', best.names)

## Validate on the TEST split

In [ ]:
metrics = best.val(
    data=str(YAML_PATH),
    imgsz=1280,
    split='test',
    conf=0.001,
    iou=0.6,
    plots=True,
    save_json=True,
    project=str(ROOT/'runs'/'faces'),
    name='faces_yolo26n_v1_TEST',
    exist_ok=True,
)
print('\nOVERALL:')
print(f'  mAP50    : {metrics.box.map50:.4f}')
print(f'  mAP50-95 : {metrics.box.map:.4f}')

## Per-class metrics (Precision / Recall / AP50 / AP50-95)

In [ ]:
names = best.names
ap_idx = list(metrics.box.ap_class_index)
pos_of_class = {int(c): i for i, c in enumerate(ap_idx)}
per_class = {}
for ci in range(len(names)):
    nm = names[ci]
    if ci in pos_of_class:
        p, r, ap50, ap = metrics.box.class_result(pos_of_class[ci])
        per_class[nm] = {'class_id': int(ci), 'precision': float(p), 'recall': float(r),
                         'ap50': float(ap50), 'ap50_95': float(ap), 'present': True}
    else:
        per_class[nm] = {'class_id': int(ci), 'precision': None, 'recall': None,
                         'ap50': None, 'ap50_95': None, 'present': False}

def _f(v): return f'{v:9.4f}' if v is not None else f"{'n/a':>9s}"
print(f"  {'class':16s} {'Precision':>9s} {'Recall':>9s} {'AP50':>9s} {'AP50-95':>9s}")
print('  ' + '-'*58)
for nm, m in per_class.items():
    print(f"  {nm:16s} {_f(m['precision'])} {_f(m['recall'])} {_f(m['ap50'])} {_f(m['ap50_95'])}")
print('  ' + '-'*58)
print(f"  {'OVERALL (mAP)':16s} {'':>9s} {'':>9s} {metrics.box.map50:9.4f} {metrics.box.map:9.4f}")

## Target-metric check (PASS / FAIL)

In [ ]:
TARGETS = {'face': {'recall': 0.95, 'precision': 0.85},
           'license-plate': {'recall': 0.90, 'precision': 0.90}}
LP_PROD_RECALL = 0.98

def badge(ok): return 'PASS' if ok else 'FAIL'
focus = {}
overall_pass = True
print('='*56)
for cname, thr in TARGETS.items():
    m = per_class.get(cname)
    print(f'\n=== {cname.upper()} ===')
    if not m or not m['present'] or m['recall'] is None:
        print('  not present in test split -> n/a'); overall_pass = False
        focus[cname] = {'present': False}; continue
    r, p = m['recall'], m['precision']
    r_ok, p_ok = r >= thr['recall'], p >= thr['precision']
    mvp = r_ok and p_ok; overall_pass = overall_pass and mvp
    print(f"  Recall    : {r:.4f}  (>= {thr['recall']:.2f})  {badge(r_ok)}")
    print(f"  Precision : {p:.4f}  (>= {thr['precision']:.2f})  {badge(p_ok)}")
    print(f'  -> MVP {cname}: {badge(mvp)}')
    e = {'present': True, 'recall': float(r), 'precision': float(p),
         'recall_threshold': thr['recall'], 'precision_threshold': thr['precision'],
         'recall_pass': bool(r_ok), 'precision_pass': bool(p_ok), 'mvp_pass': bool(mvp)}
    if cname == 'license-plate':
        e['prod_recall_threshold'] = LP_PROD_RECALL
        e['prod_recall_pass'] = bool(r >= LP_PROD_RECALL)
        print(f'  [prod goal] Recall >= {LP_PROD_RECALL:.2f}: {badge(r >= LP_PROD_RECALL)}')
    focus[cname] = e
print('\n' + '='*56)
print(f'OVERALL MVP (all focus classes pass): {badge(overall_pass)}')
print('='*56)

## Persist report (JSON + Markdown)

In [ ]:
import json
from datetime import datetime
REP_J = RUN_DIR/'eval_report.json'
REP_M = RUN_DIR/'eval_report.md'
report = {
    'run': 'faces_yolo26n_v1', 'model': 'yolo26n', 'checkpoint': str(CKPT),
    'split': 'test', 'imgsz': 1280, 'conf': 0.001, 'iou': 0.6,
    'generated_at': datetime.now().isoformat(timespec='seconds'),
    'overall': {'map50': float(metrics.box.map50), 'map50_95': float(metrics.box.map)},
    'per_class': per_class,
    'targets': {'focus': focus, 'overall_mvp_pass': bool(overall_pass),
                'note': 'In case of doubt, recall > precision (false negatives are worse).'},
}
REP_J.write_text(json.dumps(report, indent=2))

def md(v): return f'{v:.4f}' if isinstance(v, (int, float)) else 'n/a'
L = ['# Evaluation report — faces_yolo26n_v1 (TEST split)', '',
     f'- Checkpoint: `{CKPT}`', f'- Generated: {report["generated_at"]}',
     f'- Overall mAP50: **{md(metrics.box.map50)}**  |  mAP50-95: **{md(metrics.box.map)}**', '',
     '## Per-class metrics', '', '| Class | Precision | Recall | AP50 | AP50-95 |', '|---|---|---|---|---|']
for nm, m in per_class.items():
    L.append(f"| {nm} | {md(m['precision'])} | {md(m['recall'])} | {md(m['ap50'])} | {md(m['ap50_95'])} |")
L += ['', '## Target check', '', f'**Overall MVP (all focus classes pass): {"PASS" if overall_pass else "FAIL"}**', '']
for cname, e in focus.items():
    L.append(f'### {cname}')
    if not e.get('present'):
        L += ['- not present in test split -> n/a', '']; continue
    L.append(f"- Recall: {md(e['recall'])} (>= {e['recall_threshold']:.2f}) -> {'PASS' if e['recall_pass'] else 'FAIL'}")
    L.append(f"- Precision: {md(e['precision'])} (>= {e['precision_threshold']:.2f}) -> {'PASS' if e['precision_pass'] else 'FAIL'}")
    L.append(f"- **MVP {cname}: {'PASS' if e['mvp_pass'] else 'FAIL'}**")
    if 'prod_recall_pass' in e:
        L.append(f"- prod goal Recall >= {e['prod_recall_threshold']:.2f}: {'PASS' if e['prod_recall_pass'] else 'FAIL'}")
    L.append('')
L.append('> In case of doubt, recall > precision.')
REP_M.write_text('\n'.join(L) + '\n')
print('wrote:', REP_J); print('wrote:', REP_M)

## Output artifact paths

In [ ]:
outdir = ROOT/'runs'/'faces'/'faces_yolo26n_v1_TEST'
print('eval outputs in:', outdir)
for f in ['confusion_matrix.png','confusion_matrix_normalized.png','PR_curve.png','P_curve.png','R_curve.png','F1_curve.png']:
    print(('  [ok] ' if (outdir/f).exists() else '  [--] ') + f)